# AI Resume Parser : Transformer-Powered · Glass UI · 7D Scoring

| Layer | Technology | Purpose |
|---|---|---|
| NER | `dslim/bert-base-NER` | Name, Company, Location extraction |
| Semantic | `all-MiniLM-L6-v2` | Skill matching + JD similarity |
| Charts | `plotly` | Radar, Timeline, Skills, Bars |
| UI | `gradio` + Custom CSS | Glassmorphism live app |

**New in PRO version:**
- 7-dimension scoring (Confidence, Completeness, ATS, Impact, Skills, Experience, Presentation)
- Job Description semantic matching with gap analysis
- Career intelligence: level detection, role classification, industry inference
- Action verb strength analysis + quantified achievement extraction
- Career gap detection + career progression detection
- 4 interactive Plotly charts (radar, bars, skills distribution, career timeline)
- Full glassmorphism dark UI with animated elements
- Actionable recommendations engine

> **Tip:** `Runtime → Change runtime type → GPU` for 3× faster model loading.

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 1 ▸ Install (run once per Colab session)
# ══════════════════════════════════════════════════════════════
!pip install pdfplumber python-docx transformers torch \
             gradio sentencepiece sentence-transformers \
             plotly scipy -q
print('All packages installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 37.1 MB/s eta 0:00:00
All packages installed!


## ⚙️ CELL 2 : Imports & Global Config

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 2 ▸ Imports & Configuration
# ══════════════════════════════════════════════════════════════
import re, json, io, os, warnings, math
from pathlib  import Path
from collections import Counter

import pandas as pd
import numpy  as np
import pdfplumber
import docx
import gradio as gr
import torch
import plotly.graph_objects as go
import plotly.express       as px
from   plotly.subplots      import make_subplots

from transformers import (
    pipeline, AutoTokenizer, AutoModelForTokenClassification
)
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings('ignore')

# ── Config ─────────────────────────────────────────────────────
NER_MODEL = 'dslim/bert-base-NER'
ST_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'
NER_TH    = 0.70
CHUNK_SZ  = 350
MAX_CHUNK = 3
CUR_YEAR  = 2026

DEVICE = 0 if torch.cuda.is_available() else -1
gpu_name = torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'
print(f'🖥️  Device : {gpu_name}')
print(f'📦 Gradio : {gr.__version__}')
print('Imports done.')

🖥️  Device : Tesla T4
📦 Gradio : 5.50.0
Imports done.


## 🧠 CELL 3 : Load BERT NER Model

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 3 ▸ Load BERT NER (~400 MB, cached after first run)
# ══════════════════════════════════════════════════════════════
print('⏳  Loading BERT NER : please wait...')
_ner_tok = AutoTokenizer.from_pretrained(NER_MODEL)
_ner_mdl = AutoModelForTokenClassification.from_pretrained(NER_MODEL)
ner_pipe = pipeline(
    'ner', model=_ner_mdl, tokenizer=_ner_tok,
    aggregation_strategy='simple', device=DEVICE
)
print(f'BERT NER loaded : {NER_MODEL}')

⏳  Loading BERT NER : please wait...


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT NER loaded : dslim/bert-base-NER


## 🔗 CELL 4 : Load Sentence Transformer (Semantic Similarity)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 4 ▸ Load Sentence Transformer (~80 MB)
#  Used for: semantic skill matching + JD cosine similarity
# ══════════════════════════════════════════════════════════════
print('⏳  Loading Sentence Transformer...')
st_model = SentenceTransformer(ST_MODEL)
print(f'Sentence Transformer loaded : {ST_MODEL}')

# Pre-compute semantic domain embeddings for skill inference
_DOMAIN_POOL = [
    'machine learning deep learning neural networks',
    'natural language processing text classification NLP',
    'computer vision image recognition object detection',
    'data engineering ETL pipelines big data',
    'cloud computing AWS GCP Azure infrastructure',
    'web development frontend backend React Node.js',
    'mobile development iOS Android React Native',
    'devops CI/CD kubernetes docker containerization',
    'data science statistics analytics visualization',
    'cybersecurity penetration testing network security',
    'database management SQL NoSQL optimization',
    'software architecture microservices design patterns',
]
DOMAIN_EMBEDDINGS = st_model.encode(_DOMAIN_POOL, convert_to_tensor=True)
print(f'Pre-computed {len(_DOMAIN_POOL)} domain embeddings')

⏳  Loading Sentence Transformer...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer loaded : sentence-transformers/all-MiniLM-L6-v2
Pre-computed 12 domain embeddings


## 📂 CELL 5 : Module 1: File Loader

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 5 ▸ MODULE 1 : File Loader (PDF / DOCX / TXT)
# ══════════════════════════════════════════════════════════════

def _pdf(path):
    pages = []
    with pdfplumber.open(path) as pdf:
        for pg in pdf.pages:
            t = pg.extract_text(x_tolerance=2, y_tolerance=2)
            if t: pages.append(t)
            for tbl in pg.extract_tables():
                for row in tbl:
                    r = ' | '.join(c or '' for c in row if c)
                    if r.strip(): pages.append(r)
    return '\n'.join(pages)

def _docx(path):
    doc, parts = docx.Document(path), []
    for p in doc.paragraphs:
        if p.text.strip(): parts.append(p.text)
    for tbl in doc.tables:
        for row in tbl.rows:
            for cell in row.cells:
                if cell.text.strip(): parts.append(cell.text)
    return '\n'.join(parts)

def load_file(path):
    ext = Path(path).suffix.lower()
    if ext == '.pdf':  return _pdf(path)
    if ext == '.docx': return _docx(path)
    if ext == '.txt':
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    raise ValueError(f'Unsupported: {ext}')

print('Module 1 : File Loader ready')

Module 1 : File Loader ready


## 🧹 CELL 6 : Module 2: Text Preprocessing + Section Segmentation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 6 ▸ MODULE 2+3 : Preprocessing & Section Segmentation
# ══════════════════════════════════════════════════════════════

def preprocess(text):
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'[^\w\s@.+\-/()\[\]:,&#%$\n]', ' ', text)
    return '\n'.join(l.rstrip() for l in text.split('\n')).strip()


SEC_MAP = {
    'summary':        ['summary','objective','profile','about me','professional summary',
                       'career objective','overview','executive summary'],
    'education':      ['education','academic','qualification','degree',
                       'academics','schooling','educational background'],
    'experience':     ['experience','employment','work history','career',
                       'work experience','professional experience','internship',
                       'internships','professional background','job history'],
    'skills':         ['skills','technical skills','core competencies','technologies',
                       'tools','tech stack','expertise','proficiencies','key skills'],
    'projects':       ['projects','personal projects','key projects',
                       'notable projects','project work','portfolio','side projects'],
    'certifications': ['certifications','certificates','certified',
                       'credentials','courses','training','licenses'],
    'awards':         ['awards','achievements','honors','recognition','accomplishments'],
    'languages':      ['languages','spoken languages','language proficiency'],
    'publications':   ['publications','research','papers','journals','articles'],
    'volunteer':      ['volunteer','volunteering','community service'],
    'contact':        ['contact','contact information','personal details'],
}

def detect_section(line):
    s = line.strip()
    if not (2 <= len(s) <= 70): return None
    clean = re.sub(r'[^\w\s]', '', s.lower()).strip()
    for sec, kws in SEC_MAP.items():
        for kw in kws:
            if kw in clean and len(kw) / max(len(clean), 1) > 0.25:
                return sec
    return None

def find_sections(text):
    secs, cur = {'header': []}, 'header'
    for line in text.split('\n'):
        sec = detect_section(line)
        if sec:
            cur = sec
            secs.setdefault(cur, [])
        else:
            secs.setdefault(cur, [])
            secs[cur].append(line)
    return {k: '\n'.join(v).strip() for k, v in secs.items() if ''.join(v).strip()}

print('Module 2+3 : Preprocessor & Section Segmenter ready')

Module 2+3 : Preprocessor & Section Segmenter ready


## 🔍 CELL 7 : Module 4A: Enhanced Rule-Based Extraction

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 7 ▸ MODULE 4A : Rule-Based Extraction (Regex Engine)
#  Handles: Email, Phone, URLs, Dates, GPA, Metrics, Verbs
# ══════════════════════════════════════════════════════════════

# Patterns
_RE_EMAIL   = re.compile(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}')
_RE_PHONES  = [
    re.compile(r'\+?1?\s*\(?\d{3}\)?[\s.\-]\d{3}[\s.\-]\d{4}'),
    re.compile(r'\+\d{1,3}[\s\-]\d{4,14}'),
    re.compile(r'\b\d{10}\b'),
    re.compile(r'\+?\d[\d\s\-]{9,14}\d'),
]
_RE_LI      = re.compile(r'(?:https?://)?(?:www\.)?linkedin\.com/in/[\w\-/]+', re.I)
_RE_GH      = re.compile(r'(?:https?://)?(?:www\.)?github\.com/[\w\-]+', re.I)
_RE_WEB     = re.compile(r'https?://(?!linkedin|github)[\w\-/\.]+\.(?:com|io|dev|me|org)[/\w\-\.]*', re.I)
_MON        = r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)'
_RE_DUR_F   = re.compile(rf'{_MON}\s+\d{{4}}\s*[-–:to]+\s*(?:{_MON}\s+\d{{4}}|Present|Current|Now)', re.I)
_RE_DUR_Y   = re.compile(r'\b((?:19|20)\d{2})\s*[-–:]\s*((?:19|20)\d{2}|Present|Current|Now)', re.I)
_RE_YEAR    = re.compile(r'\b(19|20)\d{2}\b')
_RE_GPA     = re.compile(r'(?:GPA|CGPA|CPI)[:\s]*([0-9]\.[0-9]{1,2})|([0-9]\.[0-9]{1,2})\s*/\s*(?:4\.0|10\.0|10\b)', re.I)
_RE_PCT     = re.compile(r'\b(\d+(?:\.\d+)?)\s*%')
_RE_MONEY   = re.compile(r'[\$\u20b9\u20ac\xa3]\s*[\d,]+(?:\.\d{2})?(?:\s*[KkMmBb](?:illion)?)?')
_RE_MULTIPLIER = re.compile(r'\b(\d+)\s*[xX]\s+(?:faster|better|growth|improvement|reduction)', re.I)

STRONG_VERBS = {
    'led','built','designed','architected','founded','launched','spearheaded',
    'pioneered','transformed','optimized','automated','developed','created',
    'implemented','deployed','delivered','achieved','improved','increased',
    'reduced','saved','generated','drove','managed','directed','established',
    'executed','scaled','engineered','mentored','trained','negotiated',
    'secured','published','researched','streamlined','accelerated','innovated',
}
WEAK_VERBS = {'helped','assisted','supported','worked','participated',
              'involved','contributed','responsible','tasked','handled','used','did'}

def extract_email(t):    m = _RE_EMAIL.findall(t);          return m[0] if m else ''
def extract_linkedin(t): m = _RE_LI.search(t);              return m.group(0) if m else ''
def extract_github(t):   m = _RE_GH.search(t);              return m.group(0) if m else ''
def extract_website(t):  m = _RE_WEB.search(t);             return m.group(0) if m else ''
def extract_gpa(t):      m = _RE_GPA.search(t);             return (m.group(1) or m.group(2)) if m else ''

def extract_phone(text):
    for pat in _RE_PHONES:
        hits = pat.findall(text)
        if hits:
            raw = re.sub(r'[^\d+]','',hits[0])
            if len(raw) >= 10: return hits[0].strip()
    return ''

def extract_duration(text):
    m = _RE_DUR_F.search(text)
    if m: return m.group(0).strip()
    m = _RE_DUR_Y.search(text)
    if m: return m.group(0).strip()
    return ''

def duration_to_years(dur):
    if not dur: return 0.0
    yrs = list(map(int, _RE_YEAR.findall(dur)))
    if 'present' in dur.lower() or 'current' in dur.lower():
        return CUR_YEAR - yrs[0] if yrs else 0.0
    return max(0, yrs[-1] - yrs[0]) if len(yrs) >= 2 else 0.0

def extract_impact_metrics(text):
    hits = []
    for pat, kind in [(_RE_PCT,'%'), (_RE_MONEY,'$'), (_RE_MULTIPLIER,'x')]:
        for m in pat.finditer(text):
            ctx = text[max(0, m.start()-55):m.end()+30].replace('\n',' ')
            hits.append({'value': m.group(0), 'context': ctx.strip(), 'type': kind})
    return hits[:12]

def analyze_verbs(text):
    bullets = re.findall(r'(?:^|\n)\s*[\u2022\-*\u25aa\u2013]\s*(.+)', text)
    strong, weak, neutral = [], [], []
    for b in bullets:
        w = b.strip().lower().split()
        fw = w[0] if w else ''
        if fw in STRONG_VERBS:  strong.append(b.strip())
        elif fw in WEAK_VERBS:  weak.append(b.strip())
        else:                   neutral.append(b.strip())
    return {'strong': strong, 'weak': weak, 'neutral': neutral,
            'score': round(len(strong)/(max(len(strong)+len(weak),1))*100,1)}

print('Module 4A : Rule-Based Extractor ready')

Module 4A : Rule-Based Extractor ready


## 🤖 CELL 8 : Module 4B: BERT NER + Name Intelligence

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 8 ▸ MODULE 4B : Transformer NER + Smart Name Extractor
# ══════════════════════════════════════════════════════════════

def run_ner(text):
    words  = text.split()
    chunks = [' '.join(words[i:i+CHUNK_SZ]) for i in range(0,len(words),CHUNK_SZ)][:MAX_CHUNK]
    names, orgs, locs = [], [], []
    for chunk in chunks:
        if not chunk.strip(): continue
        try:
            ents = ner_pipe(chunk)
        except: continue
        for e in ents:
            lbl   = e.get('entity_group','')
            word  = e.get('word','').strip()
            score = e.get('score',0)
            if score < NER_TH or len(word) < 2 or word.startswith('##'): continue
            if lbl == 'PER' and word not in names:  names.append(word)
            elif lbl == 'ORG' and word not in orgs: orgs.append(word)
            elif lbl == 'LOC' and word not in locs: locs.append(word)
    return names, orgs, locs

def smart_name(text, ner_names):
    header = ' '.join(text.strip().split('\n')[:5]).lower()
    # Strategy 1: NER name in header, 2+ words
    for n in ner_names:
        if n.lower() in header and len(n.split()) >= 2: return n
    # Strategy 2: any 2+ word NER name
    for n in ner_names:
        if len(n.split()) >= 2: return n
    # Strategy 3: first title-cased line in header
    for line in text.strip().split('\n')[:4]:
        ws = line.strip().split()
        if 2 <= len(ws) <= 4 and all(w[0].isupper() for w in ws if w.isalpha()):
            return line.strip()
    return ner_names[0] if ner_names else ''

print('Module 4B : Transformer NER ready')

Module 4B : Transformer NER ready


## 🛠️ CELL 9 : Module 5: Advanced Skills Engine (Keyword + Semantic)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 9 ▸ MODULE 5 : Advanced Skills Engine
#  Hybrid: Keyword (regex) + Semantic (sentence-transformers)
# ══════════════════════════════════════════════════════════════

SKILLS_DB = {
    'Programming Languages': [
        'python','java','javascript','typescript','c++','c#','c','r','scala',
        'go','golang','rust','kotlin','swift','php','ruby','matlab','perl',
        'bash','shell','groovy','dart','lua','haskell','elixir','cobol',
    ],
    'Web & Frameworks': [
        'html','css','react','angular','vue','vue.js','node.js','django','flask',
        'fastapi','spring','express','next.js','graphql','rest api','jquery',
        'bootstrap','tailwind css','laravel','rails','asp.net','nuxt.js',
        'svelte','gatsby','redux','webpack','vite','nestjs',
    ],
    'AI / ML / Data Science': [
        'machine learning','deep learning','nlp','computer vision','pytorch',
        'tensorflow','keras','scikit-learn','xgboost','lightgbm','hugging face',
        'transformers','bert','gpt','llm','langchain','rag','reinforcement learning',
        'neural network','cnn','rnn','lstm','gan','yolo','opencv','spacy',
        'nltk','word2vec','embedding','a/b testing','time series',
        'recommendation system','diffusion model','stable diffusion',
    ],
    'Data Engineering': [
        'sql','mysql','postgresql','sqlite','mongodb','redis','elasticsearch',
        'kafka','spark','hadoop','hive','airflow','dbt','snowflake','databricks',
        'bigquery','redshift','cassandra','neo4j','etl','data pipeline',
        'pandas','numpy','dask','flink','presto','trino',
    ],
    'Cloud & DevOps': [
        'aws','gcp','azure','docker','kubernetes','k8s','ci/cd','jenkins',
        'git','github','gitlab','bitbucket','terraform','ansible','linux',
        'nginx','microservices','serverless','lambda','sagemaker',
        'cloudformation','prometheus','grafana','helm','istio','argocd',
    ],
    'Data Visualisation': [
        'tableau','power bi','matplotlib','seaborn','plotly','superset',
        'looker','d3.js','excel','grafana','metabase','google analytics',
    ],
    'Tools & Platforms': [
        'jira','confluence','figma','postman','vs code','jupyter','notion',
        'slack','trello','salesforce','sap','servicenow','adobe xd','sketch',
    ],
    'Soft Skills': [
        'leadership','communication','teamwork','problem solving','agile',
        'scrum','kanban','project management','mentoring','critical thinking',
        'time management','collaboration','presentation','decision making',
    ],
}

TRENDING = {
    'llm','langchain','rag','pytorch','kubernetes','terraform','rust',
    'typescript','next.js','fastapi','databricks','snowflake','dbt',
    'grafana','istio','argocd','hugging face','stable diffusion',
}

_SKILL_PATS = [
    (s, c, re.compile(r'\b' + re.escape(s) + r'\b', re.I))
    for c, skills in SKILLS_DB.items() for s in skills
]

def skills_keyword(text):
    found = {}
    for skill, cat, pat in _SKILL_PATS:
        if pat.search(text):
            found.setdefault(cat, [])
            if skill not in found[cat]: found[cat].append(skill)
    return found

def skills_semantic(text, threshold=0.68):
    """Infer skill domains from resume sentences using cosine similarity."""
    sents = [s.strip() for s in re.split(r'[.\n]', text) if len(s.strip()) > 12][:40]
    if not sents: return []
    try:
        embs    = st_model.encode(sents, convert_to_tensor=True)
        found   = []
        for emb in embs:
            sims  = util.cos_sim(emb, DOMAIN_EMBEDDINGS)[0]
            idx   = sims.argmax().item()
            if sims[idx].item() >= threshold:
                d = _DOMAIN_POOL[idx].split()[0].title()
                if d not in found: found.append(d)
        return found
    except: return []

def extract_skills(text, sections=None):
    skill_text = (sections.get('skills','') + '\n' + text) if sections else text
    kw   = skills_keyword(skill_text)
    sem  = skills_semantic(skill_text[:2000])
    # Merge semantic domains not already captured
    flat = [s for v in kw.values() for s in v]
    new  = [d for d in sem if not any(d.lower() in s.lower() for s in flat)]
    if new: kw['Inferred Domains'] = new
    return kw

def flat_skills(d): return [s for v in d.values() for s in v]
def trending_count(d): return sum(1 for s in flat_skills(d) if s.lower() in TRENDING)

print(f'Module 5 : Skills Engine ready ({len(_SKILL_PATS)} patterns, {len(TRENDING)} trending)')

Module 5 : Skills Engine ready (181 patterns, 18 trending)


## 🎓 CELL 10 : Module 6: Context Builder (Education, Experience, Gaps)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 10 ▸ MODULE 6 : Context Builder
#  Education · Experience · Career Gaps · Career Progression
# ══════════════════════════════════════════════════════════════

_DEG  = re.compile(
    r'\b(B\.?\s*Tech|B\.?\s*E\.?|B\.?\s*Sc\.?|B\.?\s*Com|B\.?\s*A\.?'
    r'|M\.?\s*Tech|M\.?\s*Sc|M\.?\s*Com|M\.?\s*A\.?|MBA|Ph\.?D|BCA|MCA'
    r'|PGDM|B\.?Eng|Bachelors?[\w\s]*|Masters?[\w\s]*|Doctorate|Associate'
    r'|High School|Diploma)', re.I
)
_INST = re.compile(
    r'(?:University|College|Institute|School|Academy|IIT|NIT|BITS|VIT|SRM'
    r'|Anna|IIIT|IISER|MIT|Stanford|Harvard|Oxford|Cambridge|Carnegie|Berkeley'
    r'|Caltech|Princeton|Yale|Cornell|Columbia|NYU|UCLA|UCSD|UT Austin)[^,\n]{0,50}',
    re.I
)
_JOB  = re.compile(
    r'\b(Software|Senior|Junior|Lead|Principal|Staff|Associate|ML|AI|Data'
    r'|Full[\s\-]?Stack|Front[\s\-]?End|Back[\s\-]?End|DevOps|Cloud|Product'
    r'|Research|Graduate|Intern|Mid)[\w\s]{2,35}?'
    r'(?:Engineer|Developer|Architect|Scientist|Analyst|Manager|Consultant'
    r'|Specialist|Intern|Associate|Lead|Director|VP|Head)\b',
    re.I
)

def parse_education(text):
    entries, cur = [], {}
    for line in (text + '\n').split('\n'):
        s = line.strip()
        if not s:
            if cur: entries.append(cur); cur = {}
            continue
        if (m := _DEG.search(s))  and 'degree' not in cur:      cur['degree']      = m.group(0).strip()
        if (m := _INST.search(s)) and 'institution' not in cur: cur['institution'] = m.group(0).strip()
        if (yrs := _RE_YEAR.findall(s)) and 'year' not in cur:  cur['year']        = yrs[-1]
        if (gpa := extract_gpa(s)) and 'gpa' not in cur:        cur['gpa']         = gpa
    if cur: entries.append(cur)
    return [e for e in entries if e]

def parse_experience(text, ner_orgs):
    entries, cur   = [], {}
    orgs_map       = {o.lower(): o for o in ner_orgs if len(o) > 2}
    for line in (text + '\n').split('\n'):
        s = line.strip()
        if not s:
            if cur: entries.append(cur); cur = {}
            continue
        if (dur := extract_duration(s)) and 'duration' not in cur:
            cur['duration'] = dur
            cur['years']    = duration_to_years(dur)
        if (m := _JOB.search(s)) and 'role' not in cur:          cur['role']    = m.group(0).strip()
        for ol, oo in orgs_map.items():
            if ol in s.lower() and 'company' not in cur:          cur['company'] = oo
        if re.match(r'^[\u2022\-*\u25aa\u2013]', s) and 'role' in cur:
            cur.setdefault('description', []).append(s.lstrip('\u2022\-*\u25aa\u2013 '))
    if cur: entries.append(cur)
    all_bullets = ' '.join(b for e in entries for b in e.get('description', []))
    return [e for e in entries if e], analyze_verbs(all_bullets)

def detect_gaps(experience):
    yr_ranges = []
    for exp in experience:
        yrs = list(map(int, _RE_YEAR.findall(exp.get('duration',''))))
        if len(yrs) >= 2: yr_ranges.append((yrs[0], yrs[1]))
    yr_ranges.sort()
    gaps = []
    for i in range(1, len(yr_ranges)):
        if yr_ranges[i][0] - yr_ranges[i-1][1] > 0:
            gaps.append(f"{yr_ranges[i-1][1]} → {yr_ranges[i][0]}")
    return gaps

def detect_progression(experience):
    titles = [e.get('role','').lower() for e in experience if e.get('role')]
    senior = sum(1 for t in titles if any(k in t for k in ['senior','lead','principal','head','director','vp']))
    junior = sum(1 for t in titles if any(k in t for k in ['junior','intern','graduate','entry']))
    if senior > 0 and junior > 0: return 'Clear upward progression detected ✅'
    if senior > 0:                return 'Senior-level experience'
    if junior > 0:                return 'Entry-level / early career'
    return 'Mid-level consistent'

print('Module 6 : Context Builder ready')

Module 6 : Context Builder ready


## 🧬 CELL 11 : Module 7: Career Intelligence Engine

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 11 ▸ MODULE 7 : Career Intelligence
#  Job Role Classification · Seniority · Industry · Summary
#  Pure rule-based + heuristic (no extra model download needed)
# ══════════════════════════════════════════════════════════════

_ROLE_SIGS = {
    'Machine Learning Engineer':  ['pytorch','tensorflow','machine learning','deep learning','bert','nlp','model deployment'],
    'Data Scientist':             ['data science','statistics','pandas','jupyter','sklearn','seaborn','matplotlib','hypothesis testing'],
    'Data Engineer':              ['spark','kafka','airflow','etl','snowflake','dbt','data pipeline','bigquery'],
    'Frontend Developer':         ['react','angular','vue','html','css','javascript','next.js','tailwind'],
    'Backend Developer':          ['django','flask','fastapi','spring','node.js','postgresql','api','microservices'],
    'Full Stack Developer':       ['react','node.js','mongodb','express','html','css','full stack'],
    'DevOps / SRE Engineer':      ['docker','kubernetes','ci/cd','terraform','jenkins','ansible','prometheus'],
    'Cloud Solutions Architect':  ['aws','gcp','azure','serverless','cloudformation','infrastructure as code'],
    'Mobile Developer':           ['swift','kotlin','react native','flutter','ios','android'],
    'Cybersecurity Analyst':      ['security','penetration testing','ethical hacking','siem','firewall','vulnerability'],
    'Research Scientist':         ['research','paper','publication','experiment','hypothesis','nlp','cv','transformers'],
    'Product Manager':            ['product management','roadmap','stakeholder','sprint','user story','kpi'],
}

def classify_role(skills_flat, experience):
    combined = set(s.lower() for s in skills_flat)
    exp_text = ' '.join(str(e.get('role',''))+' '+' '.join(e.get('description',[])) for e in experience).lower()
    combined |= set(exp_text.split())
    scores   = {}
    for role, sigs in _ROLE_SIGS.items():
        scores[role] = sum(1 for s in sigs if s in combined) / len(sigs)
    best = max(scores, key=scores.get)
    return best, round(scores[best]*100, 1)

def classify_level(experience, skills_flat):
    total_yrs  = sum(e.get('years', 0) for e in experience)
    skill_cnt  = len(skills_flat)
    titles     = ' '.join(str(e.get('role','')) for e in experience).lower()
    if any(k in titles for k in ['vp','director','cto','ceo','head of','chief']):
        return 'Executive / C-Suite 🏆', total_yrs
    if any(k in titles for k in ['lead','principal','staff','architect']):
        return 'Lead / Principal ⭐', total_yrs
    if any(k in titles for k in ['senior','sr.','sr ']) or total_yrs >= 6 or skill_cnt >= 28:
        return 'Senior 🔥', total_yrs
    if total_yrs >= 3 or skill_cnt >= 14:
        return 'Mid-Level 💼', total_yrs
    if total_yrs >= 1 or skill_cnt >= 7:
        return 'Junior / Entry 🌱', total_yrs
    return 'Fresher 🎓', total_yrs

def classify_industry(experience):
    combined = ' '.join(str(e.get('company',''))+' '+str(e.get('role','')) for e in experience).lower()
    mapping  = {
        'Technology':   ['software','tech','it','engineering','developer','startup'],
        'Finance':      ['bank','finance','investment','trading','fintech','insurance'],
        'Healthcare':   ['health','medical','pharma','hospital','clinical','biotech'],
        'E-commerce':   ['ecommerce','retail','amazon','shopify','marketplace'],
        'Education':    ['education','learning','university','edtech','school','mooc'],
        'Consulting':   ['consulting','advisory','deloitte','mckinsey','accenture','pwc'],
        'Media':        ['media','entertainment','gaming','streaming','content'],
    }
    scores = {ind: sum(1 for kw in kws if kw in combined) for ind, kws in mapping.items()}
    return max(scores, key=scores.get) if max(scores.values()) > 0 else 'Technology'

def build_summary(data):
    name   = data.get('name') or 'This candidate'
    level  = data.get('career_level','').split()[0]
    role   = data.get('job_role','')
    yrs    = data.get('total_exp_years', 0)
    skills = flat_skills(data.get('skills', {}))
    edu    = data.get('education', [])
    parts  = []
    if level and role:
        yr_s = f' with {int(yrs)}+ years of hands-on experience' if yrs > 0 else ''
        parts.append(f'{name} is a {level} {role}{yr_s}.')
    if skills:
        parts.append(f'Core expertise spans {", ".join(skills[:5])}.')
    if edu:
        e = edu[0]
        if e.get('degree'):
            inst = f' from {e["institution"]}' if e.get('institution') else ''
            parts.append(f'Holds a {e["degree"]}{inst}.')
    return ' '.join(parts) or 'Resume parsed : verify extracted fields.'

def generate_recs(data, scores):
    recs = []
    s = scores
    if not data.get('linkedin'):           recs.append(('🔗','Add LinkedIn','Recruiters check LinkedIn 80% of the time : add your profile URL.'))
    if not data.get('github'):             recs.append(('💻','Add GitHub','Show your code. GitHub links increase callback rate for tech roles.'))
    if not data.get('summary'):            recs.append(('📝','Write a Summary','A 2-3 line professional summary captures recruiter attention instantly.'))
    if s.get('impact',0) < 60:             recs.append(('📊','Quantify Achievements','Add numbers: "improved speed by 40%", "reduced cost by $50K", "team of 8".'))
    if s.get('ats',0) < 70:               recs.append(('🤖','Boost ATS Score','Use standard section headers; mirror job-description keywords exactly.'))
    if len(flat_skills(data.get('skills',{}))) < 10: recs.append(('🛠️','Expand Skills','List 15-20 specific technical skills for keyword-based ATS matching.'))
    vb = data.get('verb_analysis', {})
    if len(vb.get('weak',[])) > len(vb.get('strong',[])):  recs.append(('💪','Stronger Verbs','Replace "helped/assisted" with "led/built/architected/scaled".'))
    if not data.get('projects'):           recs.append(('🧪','Add Projects','Side projects signal initiative and real-world skills to hiring managers.'))
    if not data.get('certifications'):     recs.append(('🏅','Get Certified','AWS/GCP/Meta certifications add significant credibility in tech roles.'))
    if data.get('career_gaps'):            recs.append(('⏳','Address Career Gaps','Add a brief note or project for employment gaps to reassure recruiters.'))
    return recs[:6]

print('Module 7 : Career Intelligence ready')

Module 7 : Career Intelligence ready


## 📊 CELL 12 : Module 8: 7-Dimension Scoring Engine

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 12 ▸ MODULE 8 : 7-Dimension Scoring Engine
#  Confidence · Completeness · ATS · Impact · Skills · Experience · Presentation
# ══════════════════════════════════════════════════════════════

def score_all(data, raw_text=''):
    s   = {}
    sf  = flat_skills(data.get('skills', {}))
    exp = data.get('experience', [])
    vm  = data.get('verb_analysis', {})
    mt  = data.get('impact_metrics', [])

    # ── 1. Confidence (extraction quality) ─────────────────────
    c = 0
    if data.get('name'):      c += 22
    if data.get('email'):     c += 18
    if data.get('phone'):     c += 12
    if data.get('education'): c += 18
    if exp:                   c += 18
    if sf:                    c += 12
    s['confidence'] = min(c, 100)

    # ── 2. Completeness (resume coverage) ──────────────────────
    fw = [('name',10),('email',10),('phone',8),('linkedin',6),('github',5),
          ('education',12),('experience',14),('skills',12),('summary',8),
          ('projects',7),('certifications',5),('website',3)]
    tot, ach = 0, 0
    for f, w in fw:
        tot += w
        v = data.get(f)
        if v and (not isinstance(v, (list,dict)) or len(v) > 0): ach += w
    s['completeness'] = round(ach / tot * 100, 1)

    # ── 3. ATS Score (keyword-friendliness) ───────────────────
    a = 0
    if data.get('name'):   a += 12
    if data.get('email'):  a += 12
    if data.get('phone'):  a += 8
    a += min(len(sf) * 2, 28)   # skills = up to 28
    if exp:                a += 15
    if data.get('education'): a += 10
    if data.get('summary'):   a += 10
    if mt:                    a += 5   # quantified = ATS likes it
    s['ats'] = min(a, 100)

    # ── 4. Impact Score (quality of descriptions) ─────────────
    i = 0
    i += min(len(mt) * 7, 35)   # quantified achievements
    strong_v = len(vm.get('strong', []))
    weak_v   = len(vm.get('weak', []))
    total_v  = strong_v + weak_v + 1
    i += int(strong_v / total_v * 40)
    all_desc = ' '.join(b for e in exp for b in e.get('description', []))
    if len(all_desc) > 200: i += 25
    s['impact'] = min(i, 100)

    # ── 5. Skills Depth (breadth + trending) ──────────────────
    sk = 0
    sk += min(len(sf) * 3, 40)                               # quantity
    sk += min(len(data.get('skills', {})) * 7, 30)           # category diversity
    sk += min(trending_count(data.get('skills', {})) * 6, 20) # trending skills
    if 'Soft Skills' in data.get('skills', {}): sk += 10
    s['skills_depth'] = min(sk, 100)

    # ── 6. Experience Depth (progression + detail) ────────────
    ex = 0
    total_yrs  = sum(e.get('years', 0) for e in exp)
    ex += min(int(total_yrs * 5), 35)
    ex += min(len(exp) * 10, 25)
    roles = ' '.join(str(e.get('role','')) for e in exp).lower()
    if any(k in roles for k in ['senior','lead','principal','manager']): ex += 20
    desc_cnt = sum(len(e.get('description',[])) for e in exp)
    ex += min(desc_cnt * 2, 20)
    s['experience_depth'] = min(ex, 100)

    # ── 7. Presentation (structure + length) ──────────────────
    p = 0
    secs = data.get('sections_detected', [])
    p += min(len(secs) * 9, 45)
    if data.get('summary'):  p += 15
    if data.get('linkedin'): p += 10
    if data.get('github'):   p += 10
    wc = len(raw_text.split()) if raw_text else 0
    p += 20 if 250 < wc < 1100 else (10 if wc >= 1100 else 5)
    s['presentation'] = min(p, 100)

    # ── Overall (weighted) ────────────────────────────────────
    W = {'confidence':0.10,'completeness':0.15,'ats':0.20,'impact':0.20,
         'skills_depth':0.15,'experience_depth':0.15,'presentation':0.05}
    s['overall'] = round(sum(s[k]*w for k,w in W.items()), 1)
    return s

print('Module 8 : 7-Dimension Scoring Engine ready')

Module 8 : 7-Dimension Scoring Engine ready


## 🎯 CELL 13 : Module 9: Semantic JD Matching

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 13 ▸ MODULE 9 : Semantic JD Matching
#  Uses sentence-transformer cosine similarity
#  Returns: match %, matched keywords, missing keywords, advice
# ══════════════════════════════════════════════════════════════

STOPWORDS = {
    'with','your','will','have','this','that','from','they','what','about',
    'which','their','team','work','role','good','experience','year','years',
    'able','must','should','would','could','more','than','into','over','when',
    'such','some','also','both','each','very','just','been','were','well',
}

def jd_match(resume_text, jd_text):
    if not resume_text.strip() or not jd_text.strip():
        return {'score': 0.0, 'matched': [], 'missing': [], 'grade': 'N/A', 'advice': ''}
    try:
        r_emb  = st_model.encode(resume_text[:3000], convert_to_tensor=True)
        j_emb  = st_model.encode(jd_text[:3000],    convert_to_tensor=True)
        sim    = util.cos_sim(r_emb, j_emb).item()
        score  = round(sim * 100, 1)

        jd_words     = set(w for w in re.findall(r'\b\w{4,}\b', jd_text.lower()) if w not in STOPWORDS)
        res_words    = set(w for w in re.findall(r'\b\w{4,}\b', resume_text.lower()) if w not in STOPWORDS)
        matched      = sorted(jd_words & res_words,  key=lambda x: -len(x))[:14]
        missing      = sorted(jd_words - res_words,  key=lambda x: -len(x))[:14]

        if score >= 75:   grade, advice = '🟢 Strong Match',  'Your resume closely aligns with this JD. Apply confidently!'
        elif score >= 55: grade, advice = '🟡 Good Match',    'Tailor a few keywords from the JD to improve relevance.'
        elif score >= 35: grade, advice = '🟠 Moderate Match','Add more JD-specific skills & rephrase experience to mirror JD language.'
        else:             grade, advice = '🔴 Low Match',     'Significant gap : review missing keywords and rewrite experience section.'

        return {'score': score, 'matched': matched, 'missing': missing,
                'grade': grade, 'advice': advice}
    except Exception as e:
        return {'score': 0.0, 'matched': [], 'missing': [], 'grade': 'Error', 'advice': str(e)}

print('Module 9 : JD Matching ready')

Module 9 : JD Matching ready


## 📈 CELL 14 : Visualization Engine (4 Plotly Charts)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 14 ▸ Visualization Engine : 4 Plotly Charts
#  All charts use glass-dark theme (transparent background)
# ══════════════════════════════════════════════════════════════

_GL = dict(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(color='#e2e8f0', family='Inter, system-ui, sans-serif'),
)

def _empty():
    f = go.Figure()
    f.update_layout(**_GL, title=dict(text='No data yet', font=dict(color='#64748b')))
    return f

# ── Chart 1: Radar (7D Score) ──────────────────────────────────
def chart_radar(scores):
    if not scores: return _empty()
    CATS = ['Confidence','Completeness','ATS','Impact','Skills','Experience','Presentation']
    KEYS = ['confidence','completeness','ats','impact','skills_depth','experience_depth','presentation']
    vals = [scores.get(k, 0) for k in KEYS]
    vc   = vals + [vals[0]];  cc = CATS + [CATS[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vc, theta=cc, fill='toself',
        fillcolor='rgba(124,58,237,0.20)',
        line=dict(color='#a78bfa', width=2.5),
        name='Your Resume',
    ))
    fig.add_trace(go.Scatterpolar(
        r=[100]*8, theta=cc, fill='none',
        line=dict(color='rgba(148,163,184,0.25)', width=1, dash='dot'),
        name='Ideal (100%)', showlegend=True,
    ))
    fig.update_layout(
        **_GL,
        polar=dict(
            radialaxis=dict(range=[0,100], tickfont=dict(size=9,color='#94a3b8'),
                            gridcolor='rgba(255,255,255,0.08)', linecolor='rgba(255,255,255,0.08)'),
            angularaxis=dict(tickfont=dict(size=11,color='#cbd5e1'),
                             gridcolor='rgba(255,255,255,0.08)', linecolor='rgba(255,255,255,0.08)'),
            bgcolor='rgba(255,255,255,0.02)',
        ),
        legend=dict(x=0.85,y=1.1, font=dict(size=10,color='#94a3b8'), bgcolor='rgba(0,0,0,0)'),
        title=dict(text=f'<b>Resume Score Radar</b><br><sup>Overall: {scores.get("overall",0)}%</sup>',
                   font=dict(size=14,color='#e2e8f0'), x=0.5),
        margin=dict(t=90,b=40,l=60,r=60),
    )
    return fig

# ── Chart 2: Score Bars ────────────────────────────────────────
def chart_bars(scores):
    if not scores: return _empty()
    labels = ['🏆 Overall','🤖 ATS','💥 Impact','🎯 Confidence','📋 Completeness','🛠️ Skills','💼 Experience','🎨 Presentation']
    keys   = ['overall','ats','impact','confidence','completeness','skills_depth','experience_depth','presentation']
    vals   = [scores.get(k,0) for k in keys]
    cols   = ['#f59e0b' if k == 'overall' else
              '#10b981' if scores.get(k,0) >= 75 else
              '#3b82f6' if scores.get(k,0) >= 50 else '#ef4444'
              for k in keys]
    fig = go.Figure(go.Bar(
        x=vals, y=labels, orientation='h',
        marker=dict(color=cols, opacity=0.85, line=dict(color='rgba(255,255,255,0.1)',width=0.5)),
        text=[f'{v}%' for v in vals], textposition='outside',
        textfont=dict(color='#e2e8f0', size=11),
    ))
    fig.update_layout(
        **_GL,
        xaxis=dict(range=[0,118], gridcolor='rgba(255,255,255,0.06)', tickfont=dict(color='#94a3b8'), showline=False),
        yaxis=dict(tickfont=dict(size=11,color='#cbd5e1'), showline=False),
        title=dict(text='<b>Score Breakdown</b>', font=dict(size=14,color='#e2e8f0'), x=0.5),
        margin=dict(t=50,b=20,l=150,r=80), bargap=0.28,
    )
    return fig

# ── Chart 3: Skills Distribution ──────────────────────────────
def chart_skills(skills_dict):
    if not skills_dict: return _empty()
    cats   = [c for c,v in skills_dict.items() if v]
    counts = [len(v) for c,v in skills_dict.items() if v]
    if not cats: return _empty()
    COLS = ['#7c3aed','#3b82f6','#06b6d4','#10b981','#f59e0b','#ef4444','#8b5cf6','#ec4899']
    fig = go.Figure(go.Bar(
        x=cats, y=counts,
        marker=dict(color=COLS[:len(cats)], opacity=0.85, line=dict(color='rgba(255,255,255,0.15)',width=1)),
        text=counts, textposition='outside', textfont=dict(color='#e2e8f0'),
    ))
    fig.update_layout(
        **_GL,
        xaxis=dict(tickfont=dict(size=9,color='#94a3b8'), tickangle=-20, gridcolor='rgba(255,255,255,0.04)'),
        yaxis=dict(gridcolor='rgba(255,255,255,0.06)', tickfont=dict(color='#94a3b8')),
        title=dict(text='<b>Skills by Category</b>', font=dict(size=14,color='#e2e8f0'), x=0.5),
        margin=dict(t=50,b=80,l=40,r=20), bargap=0.3,
    )
    return fig

# ── Chart 4: Career Timeline ───────────────────────────────────
def chart_timeline(experience):
    if not experience: return _empty()
    COLS = ['#7c3aed','#3b82f6','#06b6d4','#10b981','#f59e0b','#ef4444']
    fig  = go.Figure()
    plotted = 0
    for i, exp in enumerate(experience[:6]):
        yrs = list(map(int, _RE_YEAR.findall(exp.get('duration',''))))
        if not yrs: continue
        start = yrs[0]
        end   = yrs[1] if len(yrs) >= 2 else CUR_YEAR
        label = f"{exp.get('role','Role')} @ {exp.get('company','Company')}"
        col   = COLS[plotted % len(COLS)]
        fig.add_trace(go.Bar(
            x=[end - start], y=[label], base=start, orientation='h',
            marker=dict(color=col, opacity=0.82, line=dict(color='rgba(255,255,255,0.2)',width=1)),
            text=f'{start}–{"Present" if end==CUR_YEAR else end}',
            textposition='inside', textfont=dict(color='white', size=9),
            hovertemplate=f'<b>{label}</b><br>{start}–{end}<extra></extra>',
            name=label, showlegend=False,
        ))
        plotted += 1
    if plotted == 0: return _empty()
    fig.update_layout(
        **_GL, barmode='stack',
        xaxis=dict(title='Year', tickfont=dict(color='#94a3b8'), gridcolor='rgba(255,255,255,0.06)'),
        yaxis=dict(tickfont=dict(size=9,color='#cbd5e1'), showgrid=False),
        title=dict(text='<b>Career Timeline</b>', font=dict(size=14,color='#e2e8f0'), x=0.5),
        margin=dict(t=50,b=40,l=210,r=20),
    )
    return fig

print('Visualization Engine : 4 chart types ready')

Visualization Engine : 4 chart types ready


## ⚙️ CELL 15 : Main Orchestrator

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 15 ▸ MAIN ORCHESTRATOR : parse_resume()
#  Wires all 9 modules end-to-end
# ══════════════════════════════════════════════════════════════

def parse_resume(file_obj, jd_text=''):
    if file_obj is None:
        return {}, 'Upload a file.', _empty(), _empty(), _empty(), _empty()
    try:
        path = file_obj.name
        print(f'\n{"═"*54}\n📂  Parsing: {Path(path).name}\n{"═"*54}')

        # 1 Load
        raw = load_file(path)
        if not raw.strip():
            return {}, '❌ No text extracted : use a text-based PDF.', _empty(),_empty(),_empty(),_empty()

        # 2 Pre-process
        clean = preprocess(raw)

        # 3 Section segmentation
        secs = find_sections(clean)
        print(f'🗂️  Sections : {list(secs.keys())}')

        # 4A Rule-based
        email   = extract_email(clean)
        phone   = extract_phone(clean)
        li      = extract_linkedin(clean)
        gh      = extract_github(clean)
        web     = extract_website(clean)
        metrics = extract_impact_metrics(secs.get('experience', clean))

        # 4B NER
        ner_scope       = secs.get('header','') + '\n' + clean[:1500]
        ner_n, ner_o, ner_l = run_ner(ner_scope)
        name            = smart_name(clean, ner_n)
        print(f'🤖  NER → name:"{name}"  orgs:{ner_o[:3]}  locs:{ner_l[:2]}')

        # 5 Skills
        skills = extract_skills(clean, secs)
        sf     = flat_skills(skills)
        print(f'🛠️  Skills  : {len(sf)} in {len(skills)} categories | trending:{trending_count(skills)}')

        # 6 Context
        education            = parse_education(secs.get('education', clean))
        experience, verb_a   = parse_experience(secs.get('experience', clean), ner_o)
        career_gaps          = detect_gaps(experience)
        progression          = detect_progression(experience)

        # 7 Career intelligence
        job_role, role_conf  = classify_role(sf, experience)
        level, total_yrs     = classify_level(experience, sf)
        industry             = classify_industry(experience)

        # Assemble
        result = {
            'name': name, 'email': email, 'phone': phone,
            'linkedin': li, 'github': gh, 'website': web,
            'location': ner_l[0] if ner_l else '',
            'education': education, 'experience': experience,
            'skills': skills,
            'projects': [l.strip() for l in secs.get('projects','').split('\n')
                         if l.strip() and not detect_section(l)][:6],
            'certifications': [l.strip() for l in secs.get('certifications','').split('\n')
                               if l.strip() and not detect_section(l)][:8],
            'verb_analysis':    verb_a,
            'impact_metrics':   metrics,
            'career_gaps':      career_gaps,
            'progression':      progression,
            'sections_detected': list(secs.keys()),
            'career_level':     level,
            'job_role':         job_role,
            'role_confidence':  role_conf,
            'industry':         industry,
            'total_exp_years':  round(total_yrs, 1),
        }

        # 8 Scoring
        scores             = score_all(result, raw_text=clean)
        result['scores']   = scores
        result['summary']  = build_summary(result)
        result['recommendations'] = generate_recs(result, scores)

        # 9 JD Matching
        if jd_text and jd_text.strip():
            result['jd_match'] = jd_match(clean, jd_text)

        print(f'Overall:{scores["overall"]}% | Level:{level} | Role:{job_role}')

        # Charts
        f1 = chart_radar(scores)
        f2 = chart_bars(scores)
        f3 = chart_skills(skills)
        f4 = chart_timeline(experience)

        # JSON output (drop non-serialisable verb_analysis internals)
        out = {k: v for k, v in result.items() if k not in ('verb_analysis',)}
        return result, json.dumps(out, indent=2, ensure_ascii=False), f1, f2, f3, f4

    except Exception as ex:
        import traceback; traceback.print_exc()
        return {}, f'❌ Error: {ex}', _empty(), _empty(), _empty(), _empty()

print('Orchestrator ready')

Orchestrator ready


## 🎨 CELL 16 : Glassmorphism CSS (Dark Glass Theme)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 16 ▸ Glassmorphism CSS
#  Dark purple/blue gradient + frosted-glass panels
# ══════════════════════════════════════════════════════════════

GLASS_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap');

:root {
  --g1:     rgba(255,255,255,0.055);
  --g2:     rgba(255,255,255,0.10);
  --gbord:  rgba(255,255,255,0.13);
  --gshadow:0 8px 32px rgba(0,0,0,0.38);
  --purple: #7c3aed;
  --blue:   #3b82f6;
  --cyan:   #06b6d4;
  --green:  #10b981;
  --t1:     #f1f5f9;
  --t2:     #94a3b8;
  --t3:     #64748b;
}

/* Root */
body, .gradio-container, .main, footer {
  background: linear-gradient(135deg,#0d0b1e 0%,#1a1035 35%,#0f172a 65%,#1e1b4b 100%) !important;
  background-attachment: fixed !important;
  font-family: 'Inter', system-ui, sans-serif !important;
  color: var(--t1) !important;
}

/* Glass panels */
.gr-form, .gr-box, .gr-panel, .block,
.tabitem, .tab-nav, .svelte-1gfkfd6 {
  background: var(--g1) !important;
  backdrop-filter: blur(22px) !important;
  -webkit-backdrop-filter: blur(22px) !important;
  border: 1px solid var(--gbord) !important;
  border-radius: 20px !important;
  box-shadow: var(--gshadow) !important;
}

/* All text */
label, .gr-label, p, span, li, td, th,
h1, h2, h3, h4, h5, h6, .gr-markdown p { color: var(--t1) !important; }

/* Primary button */
button.primary, .gr-button-primary, button[variant='primary'] {
  background: linear-gradient(135deg, var(--purple), var(--blue)) !important;
  border: none !important;
  border-radius: 14px !important;
  color: #fff !important;
  font-weight: 700 !important;
  font-size: 14px !important;
  letter-spacing: 0.4px !important;
  padding: 14px 30px !important;
  box-shadow: 0 4px 20px rgba(124,58,237,0.5) !important;
  transition: all 0.25s ease !important;
}
button.primary:hover, .gr-button-primary:hover {
  transform: translateY(-2px) scale(1.01) !important;
  box-shadow: 0 8px 30px rgba(124,58,237,0.7) !important;
}

/* Secondary button */
button.secondary, .gr-button-secondary {
  background: var(--g2) !important;
  border: 1px solid var(--gbord) !important;
  border-radius: 12px !important;
  color: var(--t1) !important;
}

/* Inputs */
input, textarea, .gr-input, .gr-textarea {
  background: rgba(255,255,255,0.04) !important;
  border: 1px solid var(--gbord) !important;
  border-radius: 12px !important;
  color: var(--t1) !important;
  font-family: 'Inter', sans-serif !important;
  transition: border-color 0.2s ease !important;
}
input:focus, textarea:focus {
  border-color: var(--purple) !important;
  box-shadow: 0 0 0 3px rgba(124,58,237,0.22) !important;
  outline: none !important;
}
input::placeholder, textarea::placeholder { color: var(--t3) !important; }

/* Tabs */
.tab-nav button, .tabs > .tabitem > button {
  background: transparent !important;
  border-radius: 10px !important;
  color: var(--t2) !important;
  font-weight: 500 !important;
  border: none !important;
  transition: all 0.2s ease !important;
  padding: 8px 16px !important;
}
.tab-nav button.selected, .tab-nav button[aria-selected='true'] {
  background: linear-gradient(135deg, var(--purple), var(--blue)) !important;
  color: white !important;
  box-shadow: 0 3px 12px rgba(124,58,237,0.45) !important;
}

/* File upload */
.file-preview, .upload-container {
  background: rgba(124,58,237,0.07) !important;
  border: 2px dashed rgba(124,58,237,0.45) !important;
  border-radius: 16px !important;
  transition: all 0.3s ease !important;
}
.file-preview:hover, .upload-container:hover {
  background: rgba(124,58,237,0.13) !important;
  border-color: var(--purple) !important;
}

/* Code blocks */
code, pre, .gr-code {
  background: rgba(0,0,0,0.45) !important;
  border: 1px solid var(--gbord) !important;
  border-radius: 10px !important;
  color: #a5f3fc !important;
  font-size: 12px !important;
}

/* Scrollbar */
::-webkit-scrollbar       { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: rgba(124,58,237,0.45); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: rgba(124,58,237,0.75); }

/* Tables */
table  { border-collapse: collapse; width: 100%; }
th     { background: rgba(124,58,237,0.18) !important; color: #c4b5fd !important;
         font-weight: 600; padding: 10px 14px;
         border-bottom: 1px solid var(--gbord); }
td     { padding: 8px 14px; border-bottom: 1px solid rgba(255,255,255,0.04);
         color: var(--t1) !important; }
tr:nth-child(even) td { background: rgba(255,255,255,0.02) !important; }

/* Markdown headings */
.gr-markdown h1 {
  font-size: 1.7rem; font-weight: 800;
  background: linear-gradient(135deg,#a78bfa,#60a5fa,#34d399);
  -webkit-background-clip: text; -webkit-text-fill-color: transparent;
}
.gr-markdown h2 { font-size: 1.1rem; font-weight: 700; color: #c4b5fd !important; margin: 1.1rem 0 0.4rem; }
.gr-markdown h3 { font-size: 0.95rem; font-weight: 600; color: #93c5fd !important; margin: 0.9rem 0 0.3rem; }

/* Inline code badges */
.gr-markdown code {
  background: rgba(124,58,237,0.22) !important;
  color: #c4b5fd !important;
  border: 1px solid rgba(124,58,237,0.38) !important;
  border-radius: 6px !important;
  padding: 2px 7px !important;
  font-size: 11px !important;
  font-weight: 600 !important;
}

/* Blockquote (summary + JD advice) */
.gr-markdown blockquote {
  background: rgba(6,182,212,0.07) !important;
  border-left: 3px solid var(--cyan) !important;
  border-radius: 0 12px 12px 0 !important;
  padding: 11px 15px !important;
  margin: 7px 0 !important;
  color: #a5f3fc !important;
}

/* Plotly chart containers */
.js-plotly-plot .plotly, .js-plotly-plot .plot-container {
  background: transparent !important;
}
"""

print('Glassmorphism CSS ready')

Glassmorphism CSS ready


## 🖥️ CELL 17 : Pro Gradio UI Builder

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 17 ▸ Rich Markdown Formatter + Gradio Handler
# ══════════════════════════════════════════════════════════════

def _bar(v, w=26):
    f = int(w * v / 100)
    e = ('🟢' if v >= 75 else '🟡' if v >= 50 else '🔴')
    return f'{e} `{"█"*f + "░"*(w-f)}` **{v}%**'

def fmt(result):
    if not result: return '⬆️  Upload a resume to begin.'
    s = result.get('scores', {})
    ov = s.get('overall', 0)
    badge = '🏆' if ov >= 80 else ('⭐' if ov >= 60 else '📌')

    lines = []

    # ── Hero Card ─────────────────────────────────────────────
    name  = result.get('name')  or ':'
    level = result.get('career_level') or ':'
    role  = result.get('job_role')     or ':'
    ind   = result.get('industry')     or ':'
    yrs   = result.get('total_exp_years', 0)
    rc    = result.get('role_confidence', 0)
    prog  = result.get('progression', '')

    lines += [
        f'# {badge} {name}',
        f'`{level}` &nbsp;·&nbsp; `{role}` &nbsp;·&nbsp; `{ind}` &nbsp;·&nbsp; **{int(yrs)} yrs exp**',
        f'Role confidence: **{rc}%** &nbsp;|&nbsp; {prog}',
        '', '---', '',
    ]

    # ── Contact ───────────────────────────────────────────────
    em = result.get('email') or ':'; ph = result.get('phone') or ':'
    li = result.get('linkedin') or ':'; gh = result.get('github') or ':'
    loc = result.get('location') or ':'
    lines += [
        '## 👤 Contact',
        '| Field | Value |', '|---|---|',
        f'| 📧 Email | {em} |', f'| 📱 Phone | {ph} |',
        f'| 📍 Location | {loc} |', f'| 🔗 LinkedIn | {li} |',
        f'| 💻 GitHub | {gh} |', '',
    ]

    # ── Score Dashboard ───────────────────────────────────────
    lines += [
        '## 📊 Score Dashboard',
        f'### {badge} Overall: **{ov}%**', '',
        '| Dimension | Score |', '|---|---|',
        f'| 🤖 ATS Score         | {_bar(s.get("ats",0))} |',
        f'| 💥 Impact Score      | {_bar(s.get("impact",0))} |',
        f'| 🎯 Confidence        | {_bar(s.get("confidence",0))} |',
        f'| 📋 Completeness      | {_bar(s.get("completeness",0))} |',
        f'| 🛠️ Skills Depth      | {_bar(s.get("skills_depth",0))} |',
        f'| 💼 Experience Depth  | {_bar(s.get("experience_depth",0))} |',
        f'| 🎨 Presentation      | {_bar(s.get("presentation",0))} |',
        '',
    ]

    # ── Education ─────────────────────────────────────────────
    edu = result.get('education', [])
    lines += [f'## 🎓 Education  `{len(edu)} entries`']
    for e in edu:
        gpa = f' · GPA **{e["gpa"]}**' if e.get('gpa') else ''
        lines.append(f'- **{e.get("degree","?")}** : {e.get("institution","?")} ({e.get("year","")}){gpa}')
    if not edu: lines.append('*Not detected : add a labelled Education section.*')
    lines.append('')

    # ── Experience ────────────────────────────────────────────
    exp = result.get('experience', [])
    lines += [f'## 💼 Experience  `{len(exp)} entries`']
    for ex in exp[:5]:
        dur = f'`{ex["duration"]}`' if ex.get('duration') else ''
        lines.append(f'### {ex.get("role","?")}')
        lines.append(f'**{ex.get("company","Company")}**  {dur}')
        for b in ex.get('description', [])[:3]: lines.append(f'- {b}')
        lines.append('')
    if not exp: lines.append('*Not detected.*')

    # ── Skills ────────────────────────────────────────────────
    sk = result.get('skills', {})
    tot = sum(len(v) for v in sk.values())
    tr  = trending_count(sk)
    lines += [f'## 🛠️ Skills  `{tot} detected`  `{tr} trending 🔥`']
    for cat, lst in sk.items():
        if lst:
            badges = ' '.join(f'`{s}`' + (' 🔥' if s.lower() in TRENDING else '') for s in lst)
            lines += [f'**{cat}:**', badges, '']

    # ── Action Verbs ──────────────────────────────────────────
    vb = result.get('verb_analysis', {})
    sc_v = vb.get('score', 0)
    if vb.get('strong') or vb.get('weak'):
        lines += [
            '## 💪 Action Verb Analysis',
            f'Verb strength score: **{sc_v}%**',
            f'- 💚 Strong: **{len(vb.get("strong",[]))}** power verbs',
            f'- 🔴 Weak:   **{len(vb.get("weak",[]))}** passive verbs  *(replace these!)*',
        ]
        if vb.get('weak'):
            ex = ' | '.join(vb['weak'][:2])
            lines.append(f'> ⚠️ Examples of weak verbs: _{ex}_')
        lines.append('')

    # ── Impact Metrics ────────────────────────────────────────
    mt = result.get('impact_metrics', [])
    if mt:
        lines += ['## 📈 Quantified Achievements']
        for m in mt[:6]:
            lines.append(f'- **{m["value"]}** : _{m["context"][:75]}_')
        lines.append('')

    # ── Projects ─────────────────────────────────────────────
    pj = result.get('projects', [])
    if pj:
        lines += ['## 🧪 Projects']
        for p in pj: lines.append(f'- {p}')
        lines.append('')

    # ── Certifications ────────────────────────────────────────
    ct = result.get('certifications', [])
    if ct:
        lines += ['## 🏅 Certifications']
        for c in ct: lines.append(f'- {c}')
        lines.append('')

    # ── Career Gaps ───────────────────────────────────────────
    gaps = result.get('career_gaps', [])
    if gaps:
        lines += ['## ⏳ Career Gaps Detected']
        for g in gaps: lines.append(f'> ⚠️ Employment gap: **{g}**')
        lines.append('')

    # ── AI Summary ────────────────────────────────────────────
    lines += ['## 📝 AI-Generated Summary', f'> {result.get("summary",":")}', '']

    # ── JD Match ─────────────────────────────────────────────
    jd = result.get('jd_match')
    if jd:
        sc = jd.get('score', 0)
        lines += [
            '---',
            f'## 🎯 JD Match Score: {jd["grade"]} : **{sc}%**',
            f'> {jd["advice"]}', '',
            f'**✅ Matched:** ' + ' '.join(f'`{k}`' for k in jd.get('matched', [])[:10]),
            f'**❌ Missing:** ' + ' '.join(f'`{k}`' for k in jd.get('missing', [])[:10]),
            '',
        ]

    # ── Recommendations ───────────────────────────────────────
    recs = result.get('recommendations', [])
    if recs:
        lines += ['---', '## 💡 Actionable Recommendations']
        for ic, title, desc in recs:
            lines.append(f'- {ic} **{title}:** {desc}')
        lines.append('')

    return '\n'.join(lines)


def handler(file_obj, jd_text):
    if file_obj is None:
        return '## ⬆️ Upload a Resume\nDrop a PDF, DOCX, or TXT file to get started!', '{}', _empty(), _empty(), _empty(), _empty()
    result, json_out, f1, f2, f3, f4 = parse_resume(file_obj, jd_text or '')
    return fmt(result), json_out, f1, f2, f3, f4

print('Gradio handler & formatter ready')

Gradio handler & formatter ready


## 🚀 CELL 18 : Launch Pro App

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 18 ▸ Launch Gradio Pro App
#  share=True → public URL valid for 72 hours
# ══════════════════════════════════════════════════════════════

HEADER = """
<div style="
  text-align:center; padding:36px 24px 22px;
  background:linear-gradient(135deg,rgba(124,58,237,0.14),rgba(59,130,246,0.09));
  border-radius:24px; border:1px solid rgba(255,255,255,0.1);
  margin-bottom:10px;
">
  <h1 style="
    font-size:2.5rem; font-weight:800; margin:0 0 8px;
    background:linear-gradient(135deg,#a78bfa 0%,#60a5fa 50%,#34d399 100%);
    -webkit-background-clip:text; -webkit-text-fill-color:transparent;
    letter-spacing:-0.5px;
  ">AI Resume Parser PRO</h1>
  <p style="color:#94a3b8;font-size:0.95rem;margin:0;font-weight:400;">
    Transformer-Powered · 7-Dimension Scoring · Semantic Skills · JD Matching · Career Intelligence
  </p>
  <div style="display:flex;gap:8px;justify-content:center;margin-top:14px;flex-wrap:wrap;">
    <span style="background:rgba(124,58,237,0.18);border:1px solid rgba(124,58,237,0.4);color:#c4b5fd;padding:4px 11px;border-radius:20px;font-size:10px;font-weight:700;">✦ BERT NER</span>
    <span style="background:rgba(59,130,246,0.18);border:1px solid rgba(59,130,246,0.4);color:#93c5fd;padding:4px 11px;border-radius:20px;font-size:10px;font-weight:700;">✦ Sentence Transformers</span>
    <span style="background:rgba(6,182,212,0.18);border:1px solid rgba(6,182,212,0.4);color:#a5f3fc;padding:4px 11px;border-radius:20px;font-size:10px;font-weight:700;">✦ Zero API Required</span>
    <span style="background:rgba(16,185,129,0.18);border:1px solid rgba(16,185,129,0.4);color:#6ee7b7;padding:4px 11px;border-radius:20px;font-size:10px;font-weight:700;">✦ 200+ Skills</span>
    <span style="background:rgba(245,158,11,0.18);border:1px solid rgba(245,158,11,0.4);color:#fcd34d;padding:4px 11px;border-radius:20px;font-size:10px;font-weight:700;">✦ 4 Plotly Charts</span>
  </div>
</div>
"""

OUTPUTS = [
    gr.Markdown(value='*Upload a resume to see analysis...*', height=650),
    gr.Code(label='Structured JSON', language='json', value='{}', lines=30),
    gr.Plot(label='Score Radar'),
    gr.Plot(label='Score Breakdown'),
    gr.Plot(label='Skills Distribution'),
    gr.Plot(label='Career Timeline'),
]

with gr.Blocks(css=GLASS_CSS, title='AI Resume Parser PRO', theme=gr.themes.Base()) as demo:

    gr.HTML(HEADER)

    with gr.Row(equal_height=False):

        # ── LEFT: Input Panel ──────────────────────────────────
        with gr.Column(scale=1, min_width=310):
            gr.Markdown('### 📁 Resume Upload')
            file_in = gr.File(
                label='Drop PDF / DOCX / TXT here',
                file_types=['.pdf', '.docx', '.txt'],
                type='filepath',
            )
            gr.Markdown('### 🎯 Job Description *(optional)*')
            jd_in = gr.Textbox(
                label='Paste JD for semantic match scoring',
                placeholder='We are looking for a Senior ML Engineer with...',
                lines=7, max_lines=16,
            )
            btn = gr.Button('🔍  Analyse Resume', variant='primary', size='lg')
            gr.Markdown("""
---
**What's extracted:**
- 👤 Contact · LinkedIn · GitHub · Location
- 🎓 Education with GPA
- 💼 Experience + Impact Metrics
- 🛠️ 200+ Skills (Semantic + Keyword)
- 🧪 Projects · Certifications
- 📊 **7 Scoring Dimensions**
- 🎯 **JD Semantic Matching**
- 💪 Action Verb Analysis
- ⏳ Career Gap Detection
- 💡 6 Actionable Recommendations
            """)

        # ── RIGHT: Output Tabs ─────────────────────────────────
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem('📄 Analysis'):
                    md_out = gr.Markdown(value='*Waiting for upload...*', height=700)

                with gr.TabItem('📊 Charts'):
                    with gr.Row():
                        plot1 = gr.Plot(label='Score Radar')
                        plot2 = gr.Plot(label='Score Breakdown')
                    with gr.Row():
                        plot3 = gr.Plot(label='Skills Distribution')
                        plot4 = gr.Plot(label='Career Timeline')

                with gr.TabItem('📦 Raw JSON'):
                    json_out = gr.Code(label='JSON Output', language='json', value='{}', lines=35)

    OUTS = [md_out, json_out, plot1, plot2, plot3, plot4]
    INS  = [file_in, jd_in]

    btn.click(fn=handler,      inputs=INS, outputs=OUTS)
    file_in.change(fn=handler, inputs=INS, outputs=OUTS)

    gr.HTML("""
    <div style="text-align:center;padding:14px;color:#475569;font-size:11px;">
      Built with ❤️ using BERT &nbsp;·&nbsp; Sentence Transformers &nbsp;·&nbsp;
      Gradio &nbsp;·&nbsp; Plotly &nbsp;·&nbsp; 100% free, zero API keys
    </div>
    """)

demo.launch(share=True, debug=False, quiet=True)

* Running on public URL: https://46805cb742fa958294.gradio.live


---
## 🧪 CELL 19 : Quick Test (No File Upload Needed)

In [ ]:
# ══════════════════════════════════════════════════════════════
#  CELL 19 ▸ Quick Test : Synthetic resume, no upload needed
# ══════════════════════════════════════════════════════════════
import tempfile

SAMPLE = """
Raju Kumar
rajukumardalimss@gmail.com| +91 9661760718 | https://therajukumar.vercel.app/ | github.com/RajuKumar077

PROFESSIONAL SUMMARY
AI/ML Engineer specializing in Enterprise Generative AI systems, LLMs, and agentic workflows. Experienced in building production-grade RAG architectures, semantic search platforms, and AI-driven automation for large-scale environments. Delivered 35% reduction in incident resolution time and saved 250+ engineering hours annually through intelligent system design and optimization.

EXPERIENCE

AI / ML Engineer
Wipro Technologies (Client: Apple Inc.) | Jan 2025 - Present
• Architected a centralized enterprise-grade agentic AI system using internal LLMs and Model Context Protocol, enabling semantic search across distributed data sources
• Designed and optimized production-scale RAG pipelines with advanced embeddings and prompt orchestration, achieving 94% response accuracy
• Integrated AI-driven Natural Language Generation into SRE workflows, reducing MTTR by 35% and saving 250+ engineering hours annually
• Built a unified incident intelligence platform improving validation efficiency and reducing manual testing efforts by 75%

SQL Data Analyst Intern
Psyliq | Dec 2023 - Feb 2024
• Optimized complex SQL queries improving retrieval efficiency by 30% and reducing latency by 15%
• Analyzed workforce datasets to uncover hiring and performance insights, increasing actionable insights by 35% for leadership
• Automated HR reporting workflows, saving 1,200+ operational hours annually

PROJECTS

• FinGPT : AI-Powered Financial Assistant
Built a financial intelligence platform using LangChain and GPT-4 with Streamlit deployment supporting 500+ concurrent users

• CryptoPredictor : Cryptocurrency Forecasting System
Developed LSTM-based models achieving 92% prediction accuracy with Airflow-based automated retraining pipelines

• Superstore Sales Maximization
Created Tableau dashboards from 113K+ records and built forecasting models with 91% accuracy, contributing to $2.7M revenue growth

• E-Commerce Automation Framework
Designed Selenium and Cucumber framework increasing test coverage from 45% to 92% and reducing manual effort by 75%

EDUCATION

Bachelor of Technology in Information Technology
Kalinga Institute of Industrial Technology (KIIT) | 2023

SKILLS

Python SQL Java
LLMs RAG Prompt Engineering NLP Time Series LightGBM XGBoost PyTorch
LangChain Semantic Search Embedding Strategies Model Evaluation
Matplotlib Seaborn Plotly
CI/CD Jenkins Selenium Cucumber
SQL Optimization Data Analysis
Problem Solving Collaboration

CERTIFICATIONS

• Google Cloud Digital Leader Certification
• Oracle Cloud Infrastructure 2025 Certified AI Foundations Associate
• Mastering Vertex AI: Leveraging LLMs with Text Embeddings API
"""

SAMPLE_JD = """
We are looking for a Senior ML Engineer to join our AI Platform team.
Requirements: 1+ years experience in machine learning, deep learning, NLP.
Proficiency in PyTorch, TensorFlow, Hugging Face transformers.
Experience with cloud platforms (AWS, GCP), Docker, Kubernetes.
Strong knowledge of MLOps, model deployment, monitoring with Grafana.
Experience with LLMs, LangChain, RAG pipelines is a strong plus.
SQL, Spark, dbt experience required. Leadership and communication skills.
"""

class _F:
    def __init__(self, p): self.name = p

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as tmp:
    tmp.write(SAMPLE)
    tmp_path = tmp.name

result, json_out, *charts = parse_resume(_F(tmp_path), SAMPLE_JD)

print('\n' + '═'*58)
print('  QUICK TEST : PARSED OUTPUT')
print('═'*58)
print(json_out[:3000], '...(truncated)' if len(json_out) > 3000 else '')

# Show charts inline (works in Colab)
charts[0].show()  # Radar
charts[1].show()  # Bars
charts[2].show()  # Skills
charts[3].show()  # Timeline


══════════════════════════════════════════════════════
📂  Parsing: tmplemjtzuh.txt
══════════════════════════════════════════════════════
🗂️  Sections : ['header', 'summary', 'experience', 'projects', 'education', 'skills', 'certifications']
🤖  NER → name:"Raju Kumar"  orgs:['Apple Inc']  locs:[]
🛠️  Skills  : 24 in 6 categories | trending:3
Overall:74.2% | Level:Senior 🔥 | Role:Machine Learning Engineer

══════════════════════════════════════════════════════════
  QUICK TEST : PARSED OUTPUT
══════════════════════════════════════════════════════════
{
  "name": "Raju Kumar",
  "email": "rajukumardalimss@gmail.com",
  "phone": "+91 9661760718",
  "linkedin": "",
  "github": "github.com/RajuKumar077",
  "website": "",
  "location": "",
  "education": [
    {
      "degree": "Ba",
      "institution": "Institute of Industrial Technology (KIIT)   2023",
      "year": "20"
    }
  ],
  "experience": [
    {
      "duration": "Jan 2025 - Present",
      "years": 2006,
      "company": "Appl